# Prefix Sum (Scan)

Master the parallel scan primitive -- the backbone of stream compaction, radix sort, and histogram equalization. Implement both Hillis-Steele and Brent-Kung algorithms.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/cuda-scan)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Then run the install cell below.

In [ ]:
!pip install numba

---

## Lesson examples (optional)

These are the code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.

### Lesson example: What is Prefix Sum?

In [ ]:
!pip install numba
import numpy as np
from numba import cuda
import math

# ============================================================
# Sequential scan for reference
# ============================================================
def sequential_inclusive_scan(arr):
    """CPU sequential inclusive prefix sum."""
    result = np.empty_like(arr)
    result[0] = arr[0]
    for i in range(1, len(arr)):
        result[i] = result[i-1] + arr[i]
    return result

def sequential_exclusive_scan(arr):
    """CPU sequential exclusive prefix sum."""
    result = np.empty_like(arr)
    result[0] = 0
    for i in range(1, len(arr)):
        result[i] = result[i-1] + arr[i-1]
    return result

# ============================================================
# Stream compaction example (CPU, to show the concept)
# ============================================================
def stream_compaction_cpu(data):
    """Remove zeros using exclusive scan."""
    # Step 1: Create mask
    mask = (data != 0).astype(np.int32)
    print(f"Data:           {data}")
    print(f"Mask (!=0):     {mask}")
    
    # Step 2: Exclusive scan of mask
    scan = sequential_exclusive_scan(mask)
    print(f"Exclusive scan: {scan}")
    
    # Step 3: Scatter kept elements
    total_kept = int(np.sum(mask))
    result = np.empty(total_kept, dtype=data.dtype)
    for i in range(len(data)):
        if mask[i]:
            result[scan[i]] = data[i]
    
    print(f"Compacted:      {result}")
    return result

# Demonstrate
print("=" * 50)
print("Inclusive scan")
arr = np.array([3, 1, 7, 0, 4, 1, 6, 3], dtype=np.int32)
print(f"Input:  {arr}")
print(f"Output: {sequential_inclusive_scan(arr)}")

print("\nExclusive scan")
print(f"Input:  {arr}")
print(f"Output: {sequential_exclusive_scan(arr)}")

print("\n" + "=" * 50)
print("Stream compaction (remove zeros)")
data = np.array([5, 0, 3, 7, 0, 2], dtype=np.int32)
stream_compaction_cpu(data)

# Show different scan operations
print("\n" + "=" * 50)
print("Scan generalizes beyond addition:")
print(f"Add-scan: {np.cumsum(arr)}")
print(f"Max-scan: {np.maximum.accumulate(arr)}")
print(f"Mul-scan: {np.cumprod(arr)}")

### Lesson example: Hillis-Steele Algorithm

In [ ]:
!pip install numba
import numpy as np
from numba import cuda
import numba
import math
import time

# ============================================================
# Hillis-Steele Inclusive Scan (single block, up to 1024 elements)
# ============================================================
@cuda.jit
def hillis_steele_inclusive_kernel(data, result, n):
    """Hillis-Steele inclusive prefix sum using shared memory.
    
    Uses double-buffering (2D shared array) to avoid race conditions.
    Handles up to 1024 elements in a single block.
    """
    shared = cuda.shared.array(shape=(2, 1024), dtype=numba.float32)
    tid = cuda.threadIdx.x
    
    # Load data into shared memory
    if tid < n:
        shared[0][tid] = data[tid]
    else:
        shared[0][tid] = 0.0
    cuda.syncthreads()
    
    # Hillis-Steele: double stride each step
    src = 0
    stride = 1
    while stride < n:
        dst = 1 - src
        if tid < n:
            if tid >= stride:
                shared[dst][tid] = shared[src][tid] + shared[src][tid - stride]
            else:
                shared[dst][tid] = shared[src][tid]
        cuda.syncthreads()
        src = dst
        stride *= 2
    
    # Write result
    if tid < n:
        result[tid] = shared[src][tid]


def hillis_steele_scan(data):
    """Host wrapper for Hillis-Steele inclusive scan."""
    n = len(data)
    assert n <= 1024, "Hillis-Steele single block: max 1024 elements"
    
    d_data = cuda.to_device(data.astype(np.float32))
    d_result = cuda.device_array(n, dtype=np.float32)
    
    threads = min(n, 1024)
    hillis_steele_inclusive_kernel[1, threads](d_data, d_result, n)
    
    return d_result.copy_to_host()


# ============================================================
# Test correctness
# ============================================================
print("=" * 60)
print("Hillis-Steele Inclusive Scan")
print("=" * 60)

# Small test
arr = np.array([3, 1, 7, 0, 4, 1, 6, 3], dtype=np.float32)
result = hillis_steele_scan(arr)
expected = np.cumsum(arr)

print(f"Input:    {arr}")
print(f"GPU scan: {result}")
print(f"Expected: {expected}")
print(f"Correct:  {np.allclose(result, expected)}")

# Larger test
print("\nLarger arrays:")
for n in [16, 64, 256, 512, 1024]:
    arr = np.random.randn(n).astype(np.float32)
    result = hillis_steele_scan(arr)
    expected = np.cumsum(arr)
    match = np.allclose(result, expected, rtol=1e-4)
    print(f"  N={n:>5}: {'PASS' if match else 'FAIL'}")

# ============================================================
# Work analysis
# ============================================================
print("\n" + "=" * 60)
print("Work Analysis: Hillis-Steele vs Sequential")
print("=" * 60)
for n in [8, 64, 256, 1024]:
    seq_work = n - 1
    hs_steps = math.ceil(math.log2(n))
    hs_work = n * hs_steps  # approximate
    print(f"  N={n:>5}: Sequential={seq_work:>5} ops, "
          f"Hillis-Steele~={hs_work:>6} ops ({hs_steps} steps), "
          f"overhead={hs_work/max(seq_work,1):.1f}x")

# ============================================================
# Timing on GPU
# ============================================================
print("\n" + "=" * 60)
print("Timing (1024 elements, 100 runs)")
print("=" * 60)
arr = np.random.randn(1024).astype(np.float32)

# Warm up
_ = hillis_steele_scan(arr)
cuda.synchronize()

# Time GPU
start = time.perf_counter()
for _ in range(100):
    _ = hillis_steele_scan(arr)
cuda.synchronize()
gpu_time = (time.perf_counter() - start) / 100

# Time CPU
start = time.perf_counter()
for _ in range(100):
    _ = np.cumsum(arr)
cpu_time = (time.perf_counter() - start) / 100

print(f"  GPU (Hillis-Steele): {gpu_time*1e6:.1f} us")
print(f"  CPU (numpy cumsum):  {cpu_time*1e6:.1f} us")
print(f"  Note: For small arrays, CPU may win due to kernel launch overhead.")
print(f"  Scan shines as part of larger GPU pipelines (no transfer cost).")

### Lesson example: Brent-Kung Algorithm

In [ ]:
!pip install numba
import numpy as np
from numba import cuda
import numba
import math
import time

# ============================================================
# Brent-Kung Exclusive Scan (single block)
# ============================================================
@cuda.jit
def brent_kung_exclusive_kernel(data, result, n):
    """Brent-Kung work-efficient exclusive prefix sum.
    
    Two phases:
    1. Up-sweep: parallel reduction to compute partial sums
    2. Down-sweep: distribute partial sums to produce prefix sums
    
    Requires n to be a power of 2. Handles up to 1024 elements.
    """
    shared = cuda.shared.array(1024, dtype=numba.float32)
    tid = cuda.threadIdx.x
    
    # Load into shared memory
    if tid < n:
        shared[tid] = data[tid]
    else:
        shared[tid] = 0.0
    cuda.syncthreads()
    
    # ---- Up-sweep (reduce) ----
    stride = 1
    while stride < n:
        index = (tid + 1) * 2 * stride - 1
        if index < n:
            shared[index] += shared[index - stride]
        stride *= 2
        cuda.syncthreads()
    
    # Set last element to identity (0 for addition)
    if tid == 0:
        shared[n - 1] = 0.0
    cuda.syncthreads()
    
    # ---- Down-sweep ----
    stride = n // 2
    while stride >= 1:
        index = (tid + 1) * 2 * stride - 1
        if index < n:
            temp = shared[index - stride]
            shared[index - stride] = shared[index]
            shared[index] += temp
        stride //= 2
        cuda.syncthreads()
    
    # Write result
    if tid < n:
        result[tid] = shared[tid]


@cuda.jit
def exclusive_to_inclusive_kernel(exclusive, original, inclusive, n):
    """Convert exclusive scan to inclusive: inclusive[i] = exclusive[i] + original[i]."""
    tid = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if tid < n:
        inclusive[tid] = exclusive[tid] + original[tid]


def next_power_of_2(n):
    """Round up to the next power of 2."""
    return 1 << (n - 1).bit_length()


def brent_kung_exclusive_scan(data):
    """Host wrapper: handles arbitrary sizes by padding to power of 2.
    Limited to arrays of at most 1024 elements (single block).
    """
    n = len(data)
    padded_n = next_power_of_2(n)
    assert padded_n <= 1024, (
        f"Array size {n} (padded to {padded_n}) exceeds single-block limit of 1024. "
        f"Use a multi-block scan for larger arrays."
    )
    
    # Pad with zeros
    padded = np.zeros(padded_n, dtype=np.float32)
    padded[:n] = data
    
    d_data = cuda.to_device(padded)
    d_result = cuda.device_array(padded_n, dtype=np.float32)
    
    brent_kung_exclusive_kernel[1, padded_n](d_data, d_result, padded_n)
    
    return d_result.copy_to_host()[:n]


def brent_kung_inclusive_scan(data):
    """Inclusive scan = exclusive scan + original data."""
    exclusive = brent_kung_exclusive_scan(data)
    return exclusive + data.astype(np.float32)


# ============================================================
# Test correctness
# ============================================================
print("=" * 60)
print("Brent-Kung Exclusive Scan")
print("=" * 60)

arr = np.array([3, 1, 7, 0, 4, 1, 6, 3], dtype=np.float32)
exclusive = brent_kung_exclusive_scan(arr)
expected_excl = np.concatenate([[0], np.cumsum(arr)[:-1]])
print(f"Input:     {arr}")
print(f"Exclusive: {exclusive}")
print(f"Expected:  {expected_excl}")
print(f"Correct:   {np.allclose(exclusive, expected_excl)}")

print(f"\nInclusive (from exclusive):")
inclusive = brent_kung_inclusive_scan(arr)
expected_incl = np.cumsum(arr)
print(f"Inclusive:  {inclusive}")
print(f"Expected:   {expected_incl}")
print(f"Correct:    {np.allclose(inclusive, expected_incl)}")

# Test various sizes (including non-power-of-2)
print("\n" + "=" * 60)
print("Testing various sizes (including non-power-of-2)")
print("=" * 60)
for n in [7, 16, 33, 100, 255, 512, 1000]:
    arr = np.random.randn(n).astype(np.float32)
    excl = brent_kung_exclusive_scan(arr)
    expected = np.concatenate([[0], np.cumsum(arr)[:-1]])
    match = np.allclose(excl, expected, rtol=1e-4)
    print(f"  N={n:>5} (padded to {next_power_of_2(n):>5}): {'PASS' if match else 'FAIL'}")

# ============================================================
# Stream compaction using scan
# ============================================================
print("\n" + "=" * 60)
print("Stream Compaction using Brent-Kung Scan")
print("=" * 60)

data = np.array([5, 0, 3, 7, 0, 2, 0, 8], dtype=np.float32)
mask = (data != 0).astype(np.float32)
scan = brent_kung_exclusive_scan(mask)
total = int(np.sum(mask))

compacted = np.empty(total, dtype=np.float32)
for i in range(len(data)):
    if mask[i]:
        compacted[int(scan[i])] = data[i]

print(f"Input:     {data}")
print(f"Mask:      {mask.astype(int)}")
print(f"Scan:      {scan.astype(int)}")
print(f"Compacted: {compacted}")

# ============================================================
# Compare work: Hillis-Steele vs Brent-Kung
# ============================================================
print("\n" + "=" * 60)
print("Work Comparison")
print("=" * 60)
for n in [8, 64, 256, 1024]:
    seq_work = n - 1
    hs_work = n * math.ceil(math.log2(n))
    bk_work = 2 * (n - 1)  # up-sweep + down-sweep
    print(f"  N={n:>5}: Sequential={seq_work:>5}, "
          f"Hillis-Steele={hs_work:>6}, "
          f"Brent-Kung={bk_work:>5}")

---

## Exercise: Parallel Scan and Stream Compaction


In [ ]:
import numpy as np
from numba import cuda
import numba
import math
import time

# ============================================================
# TODO 1: Hillis-Steele Inclusive Scan Kernel
# ============================================================
@cuda.jit
def hillis_steele_kernel(data, result, n):
    """Hillis-Steele inclusive prefix sum.
    Uses double-buffered shared memory.
    Handles up to 1024 elements in a single block.
    """
    shared = cuda.shared.array(shape=(2, 1024), dtype=numba.float32)
    tid = cuda.threadIdx.x
    
    # Load data into shared memory buffer 0
    # TODO: Load data[tid] into shared[0][tid] if tid < n, else 0
    
    cuda.syncthreads()
    
    # TODO: Implement Hillis-Steele algorithm
    # For each stride (1, 2, 4, 8, ...):
    #   - Read from source buffer
    #   - If tid >= stride: add value at tid-stride
    #   - Write to destination buffer
    #   - Swap source and destination
    #   - syncthreads()
    
    pass  # Replace with implementation


# ============================================================
# TODO 2: Brent-Kung Exclusive Scan Kernel
# ============================================================
@cuda.jit
def brent_kung_kernel(data, result, n):
    """Brent-Kung work-efficient exclusive prefix sum.
    Requires n to be a power of 2.
    Handles up to 1024 elements.
    """
    shared = cuda.shared.array(1024, dtype=numba.float32)
    tid = cuda.threadIdx.x
    
    # Load into shared memory
    # TODO: Load data into shared memory
    
    cuda.syncthreads()
    
    # TODO: Up-sweep (reduce phase)
    # For stride = 1, 2, 4, ..., n/2:
    #   index = (tid + 1) * 2 * stride - 1
    #   if index < n: shared[index] += shared[index - stride]
    #   syncthreads()
    
    # TODO: Set last element to 0
    
    # TODO: Down-sweep
    # For stride = n/2, n/4, ..., 1:
    #   index = (tid + 1) * 2 * stride - 1
    #   if index < n: swap and add
    #   syncthreads()
    
    pass  # Replace with implementation


# ============================================================
# TODO 3: Stream Compaction Kernels
# ============================================================
@cuda.jit
def create_mask_kernel(data, mask, n):
    """Create binary mask: 1 if element != 0, else 0."""
    tid = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    # TODO: Set mask[tid] = 1 if data[tid] != 0, else 0
    pass


@cuda.jit
def scatter_kernel(data, mask, scan_result, output, n):
    """Scatter non-zero elements to positions given by scan."""
    tid = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    # TODO: If mask[tid] == 1, write data[tid] to output[scan_result[tid]]
    pass


def next_power_of_2(n):
    return 1 << (n - 1).bit_length()


def stream_compact(data):
    """Remove zeros from data array using GPU scan.
    
    Steps:
    1. Create binary mask (non-zero = 1)
    2. Exclusive scan of mask -> write indices
    3. Scatter non-zero elements using indices
    """
    n = len(data)
    # TODO: Implement stream compaction using your scan kernels
    # Return the compacted array (zeros removed)
    pass


# ============================================================
# Testing
# ============================================================
print("Testing Hillis-Steele inclusive scan...")
# TODO: Test with [3, 1, 7, 0, 4, 1, 6, 3]
# Compare with np.cumsum()

print("\nTesting Brent-Kung exclusive scan...")
# TODO: Test with same array
# Compare with np.concatenate([[0], np.cumsum(arr)[:-1]])

print("\nTesting stream compaction...")
data = np.array([5, 0, 3, 7, 0, 2, 0, 8], dtype=np.float32)
# TODO: Test stream_compact(data)
# Expected: [5, 3, 7, 2, 8]

print("\nBenchmarking (N=1024)...")
# TODO: Time both scan algorithms for 1024 random elements

## Your tasks

Implement both Hillis-Steele and Brent-Kung scan algorithms, then use scan to perform stream compaction (removing zeros from an array).

**Requirements:**
1. Implement a Hillis-Steele inclusive scan kernel using shared memory with double buffering
2. Implement a Brent-Kung exclusive scan kernel with up-sweep and down-sweep phases
3. Implement a GPU stream compaction function that:
   - Creates a binary mask (1 for non-zero, 0 for zero)
   - Uses exclusive scan to compute write indices
   - Scatters non-zero elements to their computed positions
4. Test all implementations against numpy reference results
5. Benchmark both scan algorithms for N=1024

**Hints:**
- Use `cuda.shared.array(shape=(2, 1024), dtype=numba.float32)` for Hillis-Steele double buffer
- Remember: Brent-Kung needs power-of-2 sizes. Pad with zeros for arbitrary N.
- For stream compaction, you need three GPU operations: mask creation, scan, and scatter
- The total number of output elements equals the sum of the mask (or the last element of the inclusive scan)
- Use `cuda.syncthreads()` after every shared memory read/write phase

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/cuda-scan) for the solution and discussion.